# **Evaluating MedGemma-1.5-4B-IT for Diagnostic Recognition of Atypical Diseases: Endometriosis as a Case Study**

**This evaluation investigates whether MedGemma-1.5-4B-IT is able to recognize endometriosis consistently across both typical (reproductive) and atypical (organ-specific and extrapelvic) presentations. It tests for recognition across disease type, investigates symptom bias, and conducts knowledge probes.**

**Install dependencies**

In [ ]:
!pip install -U transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 79.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 18.5 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.18.0
    Uninstalling transformers-5.18.0:
      Successfully uninstalled transformers-5.18.0


**Run the cell below on your own device to authenticate your own Hugging Face account.**

In [ ]:
from huggingface_hub import login
login()

**Set up MedGemma-1.4-4B-IT**

In [ ]:
import torch
from transformers import (
    AutoProcessor,
    AutoModelForMultimodalLM,
    BitsAndBytesConfig,
)

model_id = "google/medgemma-1.5-4b-it"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(model_id)

model = AutoModelForMultimodalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map="auto",
)

processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.55k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

In [ ]:
def diagnostic_prompt(vignette):
    return [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": (
                        f"{vignette.strip()}\n\n"
                        "What is the most likely diagnosis? "
                        "Give one primary diagnosis and briefly explain your reasoning in no more than 3 sentences "
                        "based on the clinical presentation."
                    ),
                }
            ],
        }
    ]


In [ ]:
def run_case(vignette):
    messages = diagnostic_prompt(vignette)

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        generation = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
        )

    generation = generation[0][input_len:]

    output = processor.decode(
        generation,
        skip_special_tokens=True
    ).strip()

    print("VIGNETTE:")
    print(vignette.strip())
    print("\nMODEL OUTPUT:")
    print(output)

    return output

# Experiment 1: **Presentation Recognition**


**This experiment tests whether endometriosis recognition changes depending on whether the disease presents typically or atypically. There are 12 cases divided into three groups: reproductive and pelvic (typical), organ-specific (atypical), and extrapelvic (atypical).**


Reproductive and Pelvic (Typical)

In [ ]:
vignette = """
A 25-year-old woman presents with worsening painful menstruation,
pelvic pain, and pain during sexual intercourse.
"""

C1_output = run_case(vignette)

VIGNETTE:
A 25-year-old woman presents with worsening painful menstruation,
pelvic pain, and pain during sexual intercourse.

MODEL OUTPUT:
The most likely diagnosis is **endometriosis**.

This is supported by the patient's symptoms of worsening painful menstruation (dysmenorrhea), pelvic pain, and pain during sexual intercourse (dyspareunia). These are classic symptoms associated with endometriosis, a condition where tissue similar to the uterine lining grows outside the uterus. The worsening nature of the symptoms suggests a progressive condition.


In [ ]:
vignette = """
A 30-year-old woman presents with progressively worsening painful
menstruation and pain during sexual intercourse. She has also been
trying unsuccessfully to conceive.
"""

C2_output = run_case(vignette)

VIGNETTE:
A 30-year-old woman presents with progressively worsening painful
menstruation and pain during sexual intercourse. She has also been
trying unsuccessfully to conceive.

MODEL OUTPUT:
The most likely diagnosis is **endometriosis**.

This is supported by the patient's symptoms of progressively worsening painful menstruation (dysmenorrhea) and pain during sexual intercourse (dyspareunia), along with her difficulty conceiving (infertility). These are classic symptoms associated with endometriosis, a condition where tissue similar to the uterine lining grows outside the uterus.


In [ ]:
vignette = """
A 37-year-old woman presents with chronic pelvic pain, painful
menstruation, pain during sexual intercourse, and difficulty
becoming pregnant.
"""

C3_output = run_case(vignette)

VIGNETTE:
A 37-year-old woman presents with chronic pelvic pain, painful
menstruation, pain during sexual intercourse, and difficulty
becoming pregnant.

MODEL OUTPUT:
The most likely diagnosis is **endometriosis**.

This is supported by the patient's symptoms of chronic pelvic pain, dysmenorrhea (painful menstruation), dyspareunia (pain during intercourse), and infertility. These are classic symptoms associated with endometriosis, a condition where tissue similar to the uterine lining grows outside the uterus. The pain is often cyclical and related to her menstrual cycle, and the presence of the ectopic tissue can affect fertility.


In [ ]:
vignette = """
A 29-year-old woman presents with infertility, painful menstruation,
and deep pain during sexual intercourse.
"""

C4_output = run_case(vignette)

VIGNETTE:
A 29-year-old woman presents with infertility, painful menstruation,
and deep pain during sexual intercourse.

MODEL OUTPUT:
The most likely diagnosis is endometriosis. This is supported by the patient's symptoms of infertility, painful menstruation (dysmenorrhea), and deep pain during sexual intercourse (dyspareunia). These are classic symptoms associated with endometriosis, a condition where tissue similar to the uterine lining grows outside the uterus.

**Reasoning:** The combination of infertility, dysmenorrhea, and dyspareunia strongly suggests endometriosis. The pain associated with endometriosis often worsens during menstruation and can cause deep pelvic pain during intercourse due to inflammation and adhesions. Infertility can result from distorted pelvic anatomy or inflammation affecting ovulation and implantation.


**Organ-Specific (Atypical)**

In [ ]:
vignette = """
A 41-year-old woman experiences recurrent rectal bleeding
that consistently occurs around the time of menstruation.
"""

O1_output = run_case(vignette)

VIGNETTE:
A 41-year-old woman experiences recurrent rectal bleeding
that consistently occurs around the time of menstruation.

MODEL OUTPUT:
The most likely diagnosis is **dysmenorrhea** (painful menstruation) potentially related to **menorrhagia** (heavy menstrual bleeding) or **hematometra** (blood accumulation in the uterus).

**Reasoning:** The consistent timing of rectal bleeding with menstruation strongly suggests a gynecological origin, specifically related to the menstrual cycle. While the bleeding itself is concerning, its correlation with the menstrual period points towards a condition affecting the uterus or its associated structures, such as heavy bleeding, uterine inflammation, or potentially a structural issue like a submucosal fibroid that bleeds during menstruation. Further investigation, including a pelvic exam and potentially imaging or biopsy, would be needed to confirm the specific cause.


In [ ]:
vignette = """
A woman experiences recurrent painful bowel movements
that consistently occur during menstruation.
"""

O2_output = run_case(vignette)

VIGNETTE:
A woman experiences recurrent painful bowel movements
that consistently occur during menstruation.

MODEL OUTPUT:
The most likely diagnosis is **dysmenorrhea**, specifically **cyclic dysmenorrhea**. This is a common condition characterized by painful menstrual periods. The patient's description of recurrent painful bowel movements specifically during menstruation strongly suggests this diagnosis. The pain is directly linked to the menstrual cycle, indicating a physiological response to hormonal changes during menstruation. This is often associated with cramping in the lower abdomen, back pain, and sometimes other symptoms like fatigue or nausea.

**Reasoning:** The key feature is the consistent timing of the painful bowel movements with the menstrual cycle, indicating a cyclical pattern. This pattern is the hallmark of cyclic dysmenorrhea, which is often caused by uterine contractions or inflammation related to menstruation. While other conditions can cause painful bowel move

In [ ]:
vignette = """
A 29-year-old woman experiences recurrent urinary frequency,
urgency, and pain when urinating that become worse during menstruation.
"""

O3_output = run_case(vignette)

VIGNETTE:
A 29-year-old woman experiences recurrent urinary frequency,
urgency, and pain when urinating that become worse during menstruation.

MODEL OUTPUT:
The most likely diagnosis is **Urinary Tract Infection (UTI)**, specifically cystitis.

This is supported by the patient's symptoms of urinary frequency, urgency, and dysuria (pain when urinating), which are classic signs of a lower UTI. The worsening of symptoms during menstruation is also common in women with UTIs, potentially due to hormonal changes affecting the urinary tract or increased risk of infection during this time.


In [ ]:
vignette = """
A 44-year-old woman experiences recurrent right-sided flank pain
that consistently becomes worse during menstruation.
"""

O4_output = run_case(vignette)

VIGNETTE:
A 44-year-old woman experiences recurrent right-sided flank pain
that consistently becomes worse during menstruation.

MODEL OUTPUT:
The most likely diagnosis is **renal colic due to nephrolithiasis (kidney stone)**.

This is supported by the patient's age, gender, and the characteristic symptom of right-sided flank pain that worsens during menstruation. While menstruation can cause pelvic pain, the flank pain suggests a urinary tract origin, and kidney stones are a common cause of severe flank pain, often exacerbated by activities that increase intra-abdominal pressure, such as menstruation. The pain is typically described as sharp, severe, and may radiate to the groin.


**Extrapelvic (Atypical)**

In [ ]:
vignette = """
A woman in her early 40s experiences recurrent right-sided chest pain
radiating to her shoulder that coincides with menstruation.
"""

E1_output = run_case(vignette)

VIGNETTE:
A woman in her early 40s experiences recurrent right-sided chest pain
radiating to her shoulder that coincides with menstruation.

MODEL OUTPUT:
The most likely diagnosis is **anginal equivalent** or **vasospastic angina**.

This is likely because the patient's symptoms of right-sided chest pain radiating to the shoulder occurring during menstruation are highly suggestive of a cardiac origin, specifically related to coronary artery spasm. While other conditions like musculoskeletal pain or gastrointestinal issues could cause chest pain, the specific timing with menstruation and the radiation pattern are classic features often associated with cardiac ischemia, particularly in women who may have atypical presentations. The pain is likely a manifestation of reduced blood flow to the heart muscle during a period of increased demand or potential microvascular dysfunction related to hormonal changes.


In [ ]:
vignette = """
A 44-year-old woman experiences recurrent sudden right-sided chest pain
and shortness of breath that begin around the start of menstruation.
"""

E2_output = run_case(vignette)

VIGNETTE:
A 44-year-old woman experiences recurrent sudden right-sided chest pain
and shortness of breath that begin around the start of menstruation.

MODEL OUTPUT:
The most likely diagnosis is **Mittelschmerz**.

This is a common condition in women of reproductive age, characterized by sudden, sharp, unilateral lower abdominal or pelvic pain that occurs mid-cycle, typically around the time of ovulation. The patient's symptoms of sudden right-sided chest pain and shortness of breath starting around the time of menstruation (which is when ovulation typically occurs) are consistent with this diagnosis. The pain is often described as sharp and localized, and may be accompanied by other symptoms like nausea or fatigue. While the chest pain and shortness of breath are less typical, they can sometimes occur with severe pain or be related to the underlying hormonal changes.

**Reasoning:** The timing of the symptoms (around menstruation) strongly suggests a mid-cycle event related to ovulati

In [ ]:
vignette = """
A 33-year-old woman with a previous cesarean-section scar experiences
recurrent pain and swelling at the scar that increase during menstruation.
"""

E3_output = run_case(vignette)

VIGNETTE:
A 33-year-old woman with a previous cesarean-section scar experiences
recurrent pain and swelling at the scar that increase during menstruation.

MODEL OUTPUT:
The most likely diagnosis is **Adenomyosis**.

This is likely because the patient's symptoms of recurrent pain and swelling at the cesarean-section scar that worsen during menstruation are classic signs of adenomyosis. Adenomyosis is a condition where the tissue that normally lines the uterus (endometrium) grows into the muscular wall of the uterus (myometrium). This can cause uterine enlargement, pain, and heavy bleeding, and is often associated with previous uterine surgery like a cesarean section. The worsening of symptoms during menstruation is a hallmark feature.


In [ ]:
vignette = """
A 34-year-old woman experiences recurrent episodes of coughing up blood
that begin during menstruation and resolve afterward.
"""

E4_output = run_case(vignette)

VIGNETTE:
A 34-year-old woman experiences recurrent episodes of coughing up blood
that begin during menstruation and resolve afterward.

MODEL OUTPUT:
The most likely diagnosis is **menorrhagic cough**.

This is likely due to the increased blood volume and hormonal changes during menstruation, leading to irritation and inflammation of the airways, causing coughing up of blood. The timing of the cough with menstruation strongly suggests this connection.


### **Results**


| Presentation | Endometriosis recognized |
|---|---:|
| Canonical pelvic/reproductive | 4/4 |
| Organ-specific pelvic | 0/4 |
| Extrapelvic | 0/4 |
| Overall | 4 / 12 |


MedGemma did not identify endometriosis in any of the 8 atypical cases (extrapelvic and organ-specific).



# Experiment 2: **Single Symptom Bias**

**Since all typical endometriosis cases were correctly recognized in experiment 1, here the model is evaluated for possible symptom bias. To do that, we separately add one additional symptom to a fixed baseline case of endometriosis:**

- baseline: endomteriosis
- baseline + **diarrhea**: ?
- baseline + **urinary frequency**: ?
- baseline + **occasional tension headache**: ?


**Baseline 1**

In [ ]:
vignette = """
A 31-year-old woman experiences severe pain during menstruation
and deep pain during or after sexual intercourse.
"""

B1_baseline_output = run_case(vignette)

VIGNETTE:
A 31-year-old woman experiences severe pain during menstruation
and deep pain during or after sexual intercourse.

MODEL OUTPUT:
The most likely diagnosis is **endometriosis**.

This is likely because the patient's symptoms of severe dysmenorrhea (painful periods) and dyspareunia (painful intercourse) are classic signs of endometriosis. The pain often worsens during menstruation and can persist after intercourse, suggesting involvement of the pelvic structures.


In [ ]:
vignette = """
A 31-year-old woman experiences severe pain during menstruation,
deep pain during or after sexual intercourse, and recurrent diarrhea.
"""

B1_GI_output = run_case(vignette)

VIGNETTE:
A 31-year-old woman experiences severe pain during menstruation,
deep pain during or after sexual intercourse, and recurrent diarrhea.

MODEL OUTPUT:
The most likely diagnosis is **Irritable Bowel Syndrome (IBS)**.

This is likely because the patient's symptoms of severe menstrual pain (dysmenorrhea), deep pain during or after sexual intercourse (dyspareunia), and recurrent diarrhea are often associated with IBS, particularly the diarrhea-predominant type (IBS-D). While other conditions like endometriosis could cause dysmenorrhea and dyspareunia, the recurrent diarrhea is less typical and points more strongly towards IBS, which can be exacerbated by hormonal changes during menstruation. The combination of these specific symptoms makes IBS the most probable diagnosis based on the clinical presentation.


In [ ]:
vignette = """
A 31-year-old woman experiences severe pain during menstruation,
deep pain during or after sexual intercourse, and urinary frequency.
"""

B1_urinary_output = run_case(vignette)

VIGNETTE:
A 31-year-old woman experiences severe pain during menstruation,
deep pain during or after sexual intercourse, and urinary frequency.

MODEL OUTPUT:
Based on the symptoms of severe dysmenorrhea, deep dyspareunia, and urinary frequency, the most likely diagnosis is **Interstitial Cystitis/Bladder Pain Syndrome (IC/BPS)**.

This is likely because the symptoms are chronic and often associated with pelvic pain conditions. While endometriosis is a common cause of severe dysmenorrhea and deep dyspareunia, the urinary frequency is less typical and more suggestive of IC/BPS, which often presents with bladder-related symptoms. Further investigation, including imaging and potentially cystoscopy, would be needed to confirm the diagnosis and rule out other conditions like endometriosis or pelvic inflammatory disease.


In [ ]:
vignette = """
A 31-year-old woman experiences severe pain during menstruation,
deep pain during or after sexual intercourse, and occasional tension headaches.
"""

B1_unrelated_output = run_case(vignette)

VIGNETTE:
A 31-year-old woman experiences severe pain during menstruation,
deep pain during or after sexual intercourse, and occasional tension headaches.

MODEL OUTPUT:
Based on the symptoms of severe dysmenorrhea (painful periods), dyspareunia (painful intercourse), and tension headaches, the most likely diagnosis is **endometriosis**.

This is likely because endometriosis is a condition where tissue similar to the lining of the uterus grows outside the uterus, often causing pelvic pain, painful periods, and pain during intercourse. The tension headaches could be related to the chronic pain and stress associated with the condition.

The reasoning is that the combination of severe menstrual pain, painful intercourse, and headaches strongly points towards endometriosis as the underlying cause.


**Baseline 2**

In [ ]:
vignette = """
A 25-year-old woman presents with worsening painful menstruation
and recurrent pelvic pain.
"""

B2_baseline_output = run_case(vignette)

VIGNETTE:
A 25-year-old woman presents with worsening painful menstruation
and recurrent pelvic pain.

MODEL OUTPUT:
The most likely diagnosis is **endometriosis**.

This is likely because the patient's symptoms of worsening painful menstruation (dysmenorrhea) and recurrent pelvic pain are classic signs of endometriosis. The pain often worsens over time and can be associated with other symptoms like infertility or bowel/bladder issues, although these are not mentioned in the brief presentation. The age of the patient (25 years old) is also typical for the onset or worsening of endometriosis symptoms.


In [ ]:
vignette = """
A 25-year-old woman presents with worsening painful menstruation,
recurrent pelvic pain, and recurrent diarrhea.
"""

B2_GI_output = run_case(vignette)

VIGNETTE:
A 25-year-old woman presents with worsening painful menstruation,
recurrent pelvic pain, and recurrent diarrhea.

MODEL OUTPUT:
The most likely diagnosis is **Irritable Bowel Syndrome (IBS)**.

This is likely because the patient's symptoms of worsening painful menstruation, recurrent pelvic pain, and recurrent diarrhea are classic symptoms associated with IBS, particularly the diarrhea-predominant subtype (IBS-D). While endometriosis can cause painful menstruation and pelvic pain, the recurrent diarrhea is less typical and more suggestive of IBS. IBS is a common functional gastrointestinal disorder characterized by abdominal pain and altered bowel habits. The worsening of symptoms during menstruation is also a known feature of IBS.

**Reasoning:** The patient presents with a constellation of symptoms including worsening painful menstruation, recurrent pelvic pain, and recurrent diarrhea. These symptoms are highly suggestive of Irritable Bowel Syndrome (IBS), a common function

In [ ]:
vignette = """
A 25-year-old woman presents with worsening painful menstruation,
recurrent pelvic pain, and urinary frequency.
"""

B2_urinary_output = run_case(vignette)

VIGNETTE:
A 25-year-old woman presents with worsening painful menstruation,
recurrent pelvic pain, and urinary frequency.

MODEL OUTPUT:
The most likely diagnosis is **endometriosis**.

This is supported by the patient's symptoms of worsening painful menstruation (dysmenorrhea), recurrent pelvic pain, and urinary frequency, which are classic signs of endometriosis. The worsening nature of the pain suggests a progressive condition.


In [ ]:
vignette = """
A 25-year-old woman presents with worsening painful menstruation,
recurrent pelvic pain, and occasional tension headaches.
"""

B2_unrelated_output = run_case(vignette)

VIGNETTE:
A 25-year-old woman presents with worsening painful menstruation,
recurrent pelvic pain, and occasional tension headaches.

MODEL OUTPUT:
The most likely diagnosis is **Primary Dysmenorrhea**.

This is likely given the patient's age, worsening painful menstruation (dysmenorrhea), recurrent pelvic pain, and occasional tension headaches, which are common symptoms associated with this condition. The worsening nature of the pain suggests a possible progression or exacerbation of the underlying issue.


**Baseline 3**

In [ ]:
vignette = """
A 29-year-old woman presents with difficulty becoming pregnant
and painful menstruation.
"""

B3_baseline_output = run_case(vignette)

VIGNETTE:
A 29-year-old woman presents with difficulty becoming pregnant
and painful menstruation.

MODEL OUTPUT:
The most likely diagnosis is **endometriosis**.

This is likely because the patient's symptoms of difficulty becoming pregnant (infertility) and painful menstruation (dysmenorrhea) are classic signs of endometriosis. The pain often worsens during menstruation, and the condition can affect fertility by causing inflammation, scarring, and adhesions in the pelvic cavity.


In [ ]:
vignette = """
A 29-year-old woman presents with difficulty becoming pregnant,
painful menstruation, and recurrent diarrhea.
"""

B3_GI_output = run_case(vignette)

VIGNETTE:
A 29-year-old woman presents with difficulty becoming pregnant,
painful menstruation, and recurrent diarrhea.

MODEL OUTPUT:
The most likely diagnosis is **Irritable Bowel Syndrome (IBS)**.

This is likely because the patient's symptoms of painful menstruation (dysmenorrhea) and recurrent diarrhea are often associated with IBS, particularly the diarrhea-predominant type (IBS-D). While other conditions like endometriosis could cause painful periods, the prominent recurrent diarrhea points more strongly towards IBS, which can coexist with or mimic other conditions. The combination of gastrointestinal symptoms and menstrual symptoms makes IBS a strong possibility.


In [ ]:
vignette = """
A 29-year-old woman presents with difficulty becoming pregnant,
painful menstruation, and urinary frequency.
"""

B3_urinary_output = run_case(vignette)

VIGNETTE:
A 29-year-old woman presents with difficulty becoming pregnant,
painful menstruation, and urinary frequency.

MODEL OUTPUT:
The most likely diagnosis is **Interstitial Cystitis (IC)**.

This is likely because the patient's symptoms of painful menstruation (dysmenorrhea) and urinary frequency are classic hallmarks of IC. While difficulty becoming pregnant can be associated with various conditions, IC is a chronic inflammatory condition affecting the bladder and pelvic region, often presenting with these specific symptoms. The combination of painful periods and urinary issues strongly points towards this diagnosis.


In [ ]:
vignette = """
A 29-year-old woman presents with difficulty becoming pregnant,
painful menstruation, and occasional tension headaches.
"""

B3_unrelated_output = run_case(vignette)

VIGNETTE:
A 29-year-old woman presents with difficulty becoming pregnant,
painful menstruation, and occasional tension headaches.

MODEL OUTPUT:
The most likely diagnosis is **endometriosis**.

This is supported by the patient's symptoms of difficulty becoming pregnant (infertility), painful menstruation (dysmenorrhea), and occasional tension headaches. Endometriosis is a condition where tissue similar to the lining of the uterus grows outside the uterus, often causing pelvic pain, painful periods, and infertility. While tension headaches are not a classic symptom, they can occur in women with endometriosis.

**Reasoning:** The combination of infertility, dysmenorrhea, and pelvic pain strongly suggests endometriosis as the underlying cause. The patient's age is also typical for the onset or diagnosis of this condition.


## **Results**


| Condition | Endometriosis retained |
|---|---:|
| Baseline | 3/3 |
| + GI symptom | 0/3 |
| + Urinary symptom | 1/3 |
| + Unrelated symptom | 2/3 |



**Adding a gastrointentinal symptom shifted the model's prediction away from endometrosis. Adding urinary symptom shifted all but one predictions. Moreover, the model recognized 2 out of 3 baseline cases + unrelated symptoms cases as endometriosis.**

#Experiment 3: **Knowledge Probe**

Here, we test to see if the model is able to demonstrate basic knowledge of atypical endometriosis.


In [ ]:
def knowledge_prompt(question):
    return [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": question.strip()
                }
            ],
        }
    ]


def run_knowledge_probe(question):
    messages = knowledge_prompt(question)

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        generation = model.generate(
            **inputs,
            max_new_tokens=1024,
            do_sample=False,
        )

    generation = generation[0][input_len:]

    output = processor.decode(
        generation,
        skip_special_tokens=True
    ).strip()

    print("KNOWLEDGE PROBE:")
    print(question.strip())
    print("\nMODEL OUTPUT:")
    print(output)

    return output

**Endometriosis with Gastrointestinal Symptoms Knowledge Probe**

In [ ]:
gi_knowledge = """
Can endometriosis cause gastrointestinal or bowel symptoms?

Briefly describe the gastrointestinal symptoms or manifestations that may occur.
"""

run_knowledge_probe(gi_knowledge)

KNOWLEDGE PROBE:
Can endometriosis cause gastrointestinal or bowel symptoms?

Briefly describe the gastrointestinal symptoms or manifestations that may occur.

MODEL OUTPUT:
Yes, endometriosis can cause a variety of gastrointestinal (GI) symptoms or manifestations. These symptoms can be quite varied and may overlap with other conditions, making diagnosis sometimes challenging.

Here are some common GI symptoms associated with endometriosis:

*   **Abdominal Pain:** This is the most common symptom, often described as chronic pelvic pain that can radiate to the lower back, abdomen, or even other areas. It can be cyclical, worsening before and during periods.
*   **Diarrhea:** Some individuals experience changes in bowel habits, including diarrhea, often related to hormonal fluctuations.
*   **Constipation:** Less common than diarrhea, but some may experience constipation.
*   **Bloating:** Abdominal bloating is frequently reported.
*   **Nausea:** Nausea can sometimes accompany the pain 

"Yes, endometriosis can cause a variety of gastrointestinal (GI) symptoms or manifestations. These symptoms can be quite varied and may overlap with other conditions, making diagnosis sometimes challenging.\n\nHere are some common GI symptoms associated with endometriosis:\n\n*   **Abdominal Pain:** This is the most common symptom, often described as chronic pelvic pain that can radiate to the lower back, abdomen, or even other areas. It can be cyclical, worsening before and during periods.\n*   **Diarrhea:** Some individuals experience changes in bowel habits, including diarrhea, often related to hormonal fluctuations.\n*   **Constipation:** Less common than diarrhea, but some may experience constipation.\n*   **Bloating:** Abdominal bloating is frequently reported.\n*   **Nausea:** Nausea can sometimes accompany the pain or other GI symptoms.\n*   **Changes in bowel movements:** This can include urgency, frequency, or altered consistency.\n*   **Pain during bowel movements:** Some wo

**Endometriosis with Urinary Symptoms Knowledge Probe**

In [ ]:
urinary_knowledge = """
Can endometriosis cause urinary symptoms?

Briefly describe the urinary symptoms or manifestations that may occur.
"""

run_knowledge_probe(urinary_knowledge)

KNOWLEDGE PROBE:
Can endometriosis cause urinary symptoms?

Briefly describe the urinary symptoms or manifestations that may occur.

MODEL OUTPUT:
Yes, endometriosis can cause urinary symptoms. These symptoms can be quite varied and may include:

*   **Painful urination:** This can manifest as burning, stinging, or a feeling of discomfort when urinating.
*   **Urgency:** A sudden, strong need to urinate.
*   **Frequency:** Needing to urinate more often than usual.
*   **Interstitial cystitis/bladder pain syndrome (IC/BPS):** This is a condition characterized by chronic bladder pain, often accompanied by urgency and frequency. It's thought to be related to inflammation and immune system responses in the bladder, which can sometimes overlap with or be exacerbated by endometriosis.
*   **Urinary tract infections (UTIs):** Endometriosis may increase the risk of UTIs due to inflammation and altered immune function.

It's important to note that these symptoms can overlap with other condition

"Yes, endometriosis can cause urinary symptoms. These symptoms can be quite varied and may include:\n\n*   **Painful urination:** This can manifest as burning, stinging, or a feeling of discomfort when urinating.\n*   **Urgency:** A sudden, strong need to urinate.\n*   **Frequency:** Needing to urinate more often than usual.\n*   **Interstitial cystitis/bladder pain syndrome (IC/BPS):** This is a condition characterized by chronic bladder pain, often accompanied by urgency and frequency. It's thought to be related to inflammation and immune system responses in the bladder, which can sometimes overlap with or be exacerbated by endometriosis.\n*   **Urinary tract infections (UTIs):** Endometriosis may increase the risk of UTIs due to inflammation and altered immune function.\n\nIt's important to note that these symptoms can overlap with other conditions, so a proper diagnosis requires evaluation by a healthcare professional."

**Endometriosis with Extrapelvic Symptoms Knowledge Probe**

In [ ]:
extrapelvic_knowledge = """
Can endometriosis occur outside the pelvic cavity?

Briefly describe examples of extrapelvic manifestations of endometriosis.
"""

run_knowledge_probe(extrapelvic_knowledge)

KNOWLEDGE PROBE:
Can endometriosis occur outside the pelvic cavity?

Briefly describe examples of extrapelvic manifestations of endometriosis.

MODEL OUTPUT:
Yes, endometriosis can occur outside the pelvic cavity. This is known as **ectopic endometriosis**.

Here are some examples of extrapelvic manifestations of endometriosis:

*   **Ovarian endometriomas:** These are cysts on the ovary that contain old blood, similar to a uterine period.
*   **Uterine fossa endometriomas:** These are cysts in the space where the uterus sits.
*   **Peritoneal endometriomas:** These are cysts on the outer surface of the abdomen.
*   **Intra-abdominal endometriomas:** These are cysts within the abdominal cavity.
*   **Endometriosis in the bowel:** This can cause inflammation, scarring, and adhesions in the intestines.
*   **Endometriosis in the bladder:** This can cause inflammation, scarring, and adhesions in the bladder.
*   **Endometriosis in the lungs:** This is rare but can cause inflammation and s

"Yes, endometriosis can occur outside the pelvic cavity. This is known as **ectopic endometriosis**.\n\nHere are some examples of extrapelvic manifestations of endometriosis:\n\n*   **Ovarian endometriomas:** These are cysts on the ovary that contain old blood, similar to a uterine period.\n*   **Uterine fossa endometriomas:** These are cysts in the space where the uterus sits.\n*   **Peritoneal endometriomas:** These are cysts on the outer surface of the abdomen.\n*   **Intra-abdominal endometriomas:** These are cysts within the abdominal cavity.\n*   **Endometriosis in the bowel:** This can cause inflammation, scarring, and adhesions in the intestines.\n*   **Endometriosis in the bladder:** This can cause inflammation, scarring, and adhesions in the bladder.\n*   **Endometriosis in the lungs:** This is rare but can cause inflammation and scarring.\n*   **Endometriosis in the brain:** This is extremely rare but has been reported in association with endometriosis.\n\nIt's important to 

**Endometriosis with Thoracic Symptoms Knowledge Probe**

In [ ]:
thoracic_knowledge = """
Can endometriosis affect the chest or thoracic cavity?

Briefly describe the thoracic symptoms or manifestations that may occur.
"""

run_knowledge_probe(thoracic_knowledge)

KNOWLEDGE PROBE:
Can endometriosis affect the chest or thoracic cavity?

Briefly describe the thoracic symptoms or manifestations that may occur.

MODEL OUTPUT:
While endometriosis primarily affects the pelvic region, it's possible for the condition to extend to other areas, including the chest or thoracic cavity. This is known as **thoracic endometriosis**.

**Thoracic endometriosis** can manifest in various ways, often mimicking other conditions. Some potential symptoms or manifestations include:

*   **Chest pain:** This is a common symptom, often described as sharp, stabbing, or aching. It might be localized or radiate.
*   **Dyspnea (shortness of breath):** Difficulty breathing.
*   **Chest wall tenderness:** Pain when touching the chest wall.
*   **Pleural effusion:** Fluid buildup in the space between the lungs and chest wall.
*   **Pneumothorax:** Air leakage into the space between the lungs and chest wall.
*   **Rib pain:** Pain specifically along the ribs.
*   **Less common s

"While endometriosis primarily affects the pelvic region, it's possible for the condition to extend to other areas, including the chest or thoracic cavity. This is known as **thoracic endometriosis**.\n\n**Thoracic endometriosis** can manifest in various ways, often mimicking other conditions. Some potential symptoms or manifestations include:\n\n*   **Chest pain:** This is a common symptom, often described as sharp, stabbing, or aching. It might be localized or radiate.\n*   **Dyspnea (shortness of breath):** Difficulty breathing.\n*   **Chest wall tenderness:** Pain when touching the chest wall.\n*   **Pleural effusion:** Fluid buildup in the space between the lungs and chest wall.\n*   **Pneumothorax:** Air leakage into the space between the lungs and chest wall.\n*   **Rib pain:** Pain specifically along the ribs.\n*   **Less common symptoms:** Such as cough, fever, or even palpitations, depending on the specific location and extent of the endometriosis.\n\nIt's important to note t

### **Results**

| Knowledge Probe | Relevant Knowledge Demonstrated? |
|---|---:|
| Gastrointestinal / bowel manifestations | Yes |
| Urinary manifestations | Yes |
| Extrapelvic manifestations | Yes |
| Thoracic manifestations | Yes |

**MedGemma was able to demonstrate explicit knowledge that endometriosis can produce gastrointestinal, urinary, extrapelvic and thoracic locations.**
